In [0]:
%run "/Workspace/ETL_ARQUITETURA_MEDALHAO/00.config/config"

Catálogo: workspace
Schemas: bronze_economia silver_economia gold_economia


In [0]:


# 01 - Imports

from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, regexp_replace, col, greatest



# 02 - Ler tabelas Bronze (IPCA e Boi Gordo)

ipca = spark.table(f"{CATALOG}.{BRONZE}.ipca")
boi  = spark.table(f"{CATALOG}.{BRONZE}.boi_gordo")

print("Schema IPCA:")
ipca.printSchema()

print("Schema Boi Gordo:")
boi.printSchema()



# 03 - (Opcional) Visualização inicial

display(ipca)
display(boi)



# 04 - Renomear colunas do boi gordo (Data/Valor -> data/boi_gordo)

from pyspark.sql.functions import col

boi = (
    boi
    .withColumnRenamed("Data", "data")
    .withColumnRenamed("Valor", "boi_gordo")
)

display(boi)
boi.printSchema()



# 05 - Ajuste de datas para boi gordo (MM/yyyy -> date -> yyyy-MM como string)

from pyspark.sql.functions import to_date, date_format

# 1) transforma string 'MM/yyyy' em date (com dia default)
boi = boi.withColumn("data", to_date("data", "MM/yyyy"))

# 2) formata essa date como string 'yyyy-MM'
boi = boi.withColumn("data", date_format("data", "yyyy-MM"))

display(boi)



# 06 - Ajuste de datas para IPCA (dd/MM/yyyy -> date -> yyyy-MM como string)

ipca = ipca.withColumn("data", to_date("data", "dd/MM/yyyy"))
ipca = ipca.withColumn("data", date_format("data", "yyyy-MM"))

display(ipca)



# 07 - Primeiro join simples por 'data' (como string yyyy-MM)

df_join = (
    ipca.join(boi, on="data", how="inner")
         .select("data", "ipca", "boi_gordo")
)

display(df_join)



# 08 - Join com alias, trazendo também data_coleta do IPCA

ip = ipca.alias("ip")
bo = boi.alias("bo")

df_join = (
    ip.join(bo, col("ip.data") == col("bo.data"), "inner")
      .select(
          col("ip.data").alias("data"),
          col("ip.ipca").alias("ipca"),
          col("bo.boi_gordo").alias("boi_gordo"),
          col("ip.data_coleta").alias("data_coleta")
      )
)

display(df_join)



# 09 - Correção de tipos: data como date yyyy-MM-01, números como double

from pyspark.sql.functions import regexp_replace

# 'data' está como string 'yyyy-MM' -> vira date assumindo dia '01'
df_join = df_join.withColumn("data", to_date(col("data"), "yyyy-MM"))

# ipca e boi_gordo: garantir double (se vier com vírgula, troca por ponto)
df_join = (
    df_join
    .withColumn("ipca",      regexp_replace("ipca",      ",", ".").cast("double"))
    .withColumn("boi_gordo", regexp_replace("boi_gordo", ",", ".").cast("double"))
)

df_join.printSchema()
display(df_join.orderBy("data").limit(50))



# 09 - Salvar em tabela Delta Silver (silver_etl.economia)

df_join.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{SILVER}.economia")



Schema IPCA:
root
 |-- data: string (nullable = true)
 |-- ipca: double (nullable = true)
 |-- data_coleta: timestamp (nullable = true)

Schema Boi Gordo:
root
 |-- Data: string (nullable = true)
 |-- Valor: string (nullable = true)
 |-- data_coleta: timestamp (nullable = true)



data,ipca,data_coleta
01/01/2024,0.42,2026-05-07T01:58:45.427Z
01/02/2024,0.83,2026-05-07T01:58:45.427Z
01/03/2024,0.16,2026-05-07T01:58:45.427Z
01/04/2024,0.38,2026-05-07T01:58:45.427Z
01/05/2024,0.46,2026-05-07T01:58:45.427Z
01/06/2024,0.21,2026-05-07T01:58:45.427Z
01/07/2024,0.38,2026-05-07T01:58:45.427Z
01/08/2024,-0.02,2026-05-07T01:58:45.427Z
01/09/2024,0.44,2026-05-07T01:58:45.427Z
01/10/2024,0.56,2026-05-07T01:58:45.427Z


Data,Valor,data_coleta
01/2024,"249,65",2026-05-07T01:43:56.799Z
02/2024,"237,84",2026-05-07T01:43:56.799Z
03/2024,"232,81",2026-05-07T01:43:56.799Z
04/2024,"230,51",2026-05-07T01:43:56.799Z
05/2024,"226,92",2026-05-07T01:43:56.799Z
06/2024,"220,70",2026-05-07T01:43:56.799Z
07/2024,"229,27",2026-05-07T01:43:56.799Z
08/2024,"235,07",2026-05-07T01:43:56.799Z
09/2024,"255,45",2026-05-07T01:43:56.799Z
10/2024,"301,13",2026-05-07T01:43:56.799Z


data,boi_gordo,data_coleta
01/2024,"249,65",2026-05-07T01:43:56.799Z
02/2024,"237,84",2026-05-07T01:43:56.799Z
03/2024,"232,81",2026-05-07T01:43:56.799Z
04/2024,"230,51",2026-05-07T01:43:56.799Z
05/2024,"226,92",2026-05-07T01:43:56.799Z
06/2024,"220,70",2026-05-07T01:43:56.799Z
07/2024,"229,27",2026-05-07T01:43:56.799Z
08/2024,"235,07",2026-05-07T01:43:56.799Z
09/2024,"255,45",2026-05-07T01:43:56.799Z
10/2024,"301,13",2026-05-07T01:43:56.799Z


root
 |-- data: string (nullable = true)
 |-- boi_gordo: string (nullable = true)
 |-- data_coleta: timestamp (nullable = true)



...

data,ipca,data_coleta
2024-01,0.42,2026-05-07T01:58:45.427Z
2024-02,0.83,2026-05-07T01:58:45.427Z
2024-03,0.16,2026-05-07T01:58:45.427Z
2024-04,0.38,2026-05-07T01:58:45.427Z
2024-05,0.46,2026-05-07T01:58:45.427Z
2024-06,0.21,2026-05-07T01:58:45.427Z
2024-07,0.38,2026-05-07T01:58:45.427Z
2024-08,-0.02,2026-05-07T01:58:45.427Z
2024-09,0.44,2026-05-07T01:58:45.427Z
2024-10,0.56,2026-05-07T01:58:45.427Z


data,ipca,boi_gordo
2024-01,0.42,"249,65"
2024-02,0.83,"237,84"
2024-03,0.16,"232,81"
2024-04,0.38,"230,51"
2024-05,0.46,"226,92"
2024-06,0.21,"220,70"
2024-07,0.38,"229,27"
2024-08,-0.02,"235,07"
2024-09,0.44,"255,45"
2024-10,0.56,"301,13"


data,ipca,boi_gordo,data_coleta
2024-01,0.42,"249,65",2026-05-07T01:58:45.427Z
2024-02,0.83,"237,84",2026-05-07T01:58:45.427Z
2024-03,0.16,"232,81",2026-05-07T01:58:45.427Z
2024-04,0.38,"230,51",2026-05-07T01:58:45.427Z
2024-05,0.46,"226,92",2026-05-07T01:58:45.427Z
2024-06,0.21,"220,70",2026-05-07T01:58:45.427Z
2024-07,0.38,"229,27",2026-05-07T01:58:45.427Z
2024-08,-0.02,"235,07",2026-05-07T01:58:45.427Z
2024-09,0.44,"255,45",2026-05-07T01:58:45.427Z
2024-10,0.56,"301,13",2026-05-07T01:58:45.427Z


root
 |-- data: date (nullable = true)
 |-- ipca: double (nullable = true)
 |-- boi_gordo: double (nullable = true)
 |-- data_coleta: timestamp (nullable = true)



data,ipca,boi_gordo,data_coleta
2024-01-01,0.42,249.65,2026-05-07T01:58:45.427Z
2024-02-01,0.83,237.84,2026-05-07T01:58:45.427Z
2024-03-01,0.16,232.81,2026-05-07T01:58:45.427Z
2024-04-01,0.38,230.51,2026-05-07T01:58:45.427Z
2024-05-01,0.46,226.92,2026-05-07T01:58:45.427Z
2024-06-01,0.21,220.7,2026-05-07T01:58:45.427Z
2024-07-01,0.38,229.27,2026-05-07T01:58:45.427Z
2024-08-01,-0.02,235.07,2026-05-07T01:58:45.427Z
2024-09-01,0.44,255.45,2026-05-07T01:58:45.427Z
2024-10-01,0.56,301.13,2026-05-07T01:58:45.427Z
